**1.1 Install dependencies**

In [ ]:
!pip install -U datasets
!pip install transformers  -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.3/506.3 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 18.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.


**2. Practical Case**

In [ ]:
# ✅ Paso 1: Install Required Libraries
!pip install -q pymupdf transformers
import fitz  # PyMuPDF
from google.colab import files

from transformers import pipeline
import math
import re
import unicodedata
import pandas as pd
from IPython.display import Markdown, display


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 72.0 MB/s eta 0:00:00


In [ ]:
def clean_text(text):
    # Reemplaza caracteres invisibles o especiales
    text = text.replace("\xa0", " ")  # NBSP a espacio normal
    text = unicodedata.normalize("NFKC", text)  # Normaliza caracteres Unicode

    # Limpieza de espacios y saltos de línea
    text = re.sub(r"[ \t]+", " ", text)              # Múltiples espacios/tab por uno
    text = re.sub(r"\s*\n\s*", "\n", text)           # Saltos de línea limpios
    text = re.sub(r"\n{2,}", "\n", text)             # Evita dobles saltos

    # Elimina espacios antes de puntuación
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)

    # Corrige espacios duplicados finales
    return text.strip()

In [ ]:
# ✅ Paso 2: Cargar el contrato en pdf

uploaded = files.upload()  # Se debe seleccionar el documento en pdf

# ✅ Paso 3: Extraer el texto de PDF usando PyMupdf


filename = next(iter(uploaded))  # Automatizar la subida del archivo
doc = fitz.open(filename)

# Extraer el texto
full_text = ""
for page in doc:
    full_text += page.get_text()

print(f"✅ Extracted {len(full_text)} characters from PDF")

# ✅ Paso 4: Cargar el Summarization Model de Hugging Face



summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# ✅ Paso 5: Break long text into manageable chunks (BART max length ~1024 tokens)


def chunk_text(text, max_chunk_length=1024):
    paragraphs = text.split("\n")
    chunks = []
    current = ""

    for para in paragraphs:
        if len(current) + len(para) <= max_chunk_length:
            current += para + "\n"
        else:
            chunks.append(current)
            current = para + "\n"
    if current:
        chunks.append(current)
    return chunks

chunks = chunk_text(full_text, max_chunk_length=1024)

Saving Contrato_ejercicio (3).pdf to Contrato_ejercicio (3).pdf
✅ Extracted 10350 characters from PDF


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [ ]:
# ✅ Paso 6: Generar Summaries for Each Chunk
print(f"📚 Summarizing {len(chunks)} chunks...")
intermediate_summaries = []

for chunk in chunks:
    summary = summarizer(chunk, max_length=150, min_length=40, do_sample=False)[0]['summary_text']
    intermediate_summaries.append(summary)



📚 Summarizing 11 chunks...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
# ✅ Paso 7: Limpiar los mini-summaries antes del resumen final
cleaned_summaries = [clean_text(s) for s in intermediate_summaries]
combined = " ".join(cleaned_summaries)


executive_summary = summarizer(
    combined,
    max_length=600,
    min_length=400,
    do_sample=False
)[0]["summary_text"]

In [ ]:
executive_summary

'Los partes acuerdan que el monto de la renta que pagará EL arrestatario pagarán EL EL ARRENDATARIO. El renta será pago por mensualidades, realizándose el abono el día 11 de cada mes. El depósito deberá efectuar en la siguiente bancaria a nombre de ZAMORA QUISPE. El contrato será de 12 meses que se computará a partir of the fecha de suscripción. El bien como le será entregado, lo cual al final quedará a favor of the dueña, sin reembolso alguno. Para efectos que segenere con el contrato, las partes se someten a la competencia territorial de los jueces y tribunales del cono norte de Lima. El pago per mes de Setiembre del 2024 por la suma de S/ 1400 MIL CUATROCIENTOS SOLES a LA ARRendADORA. El barrio de Los Laureles N.o 450, Mz. B Lt. 20, Urb. Villa Central. El local comercial se encuentra desocupado en buen estado de conservación y habitabilidad, con piso de cerámica tipo porcelanato. El parte de Zamora QuISPE está en la obligación moral y en especial legal de gestionar su autorización d

In [ ]:
display(Markdown(f"### 📄 RESUMEN EJECUTIVO\n\n{executive_summary}"))

### 📄 RESUMEN EJECUTIVO

Los partes acuerdan que el monto de la renta que pagará EL arrestatario pagarán EL EL ARRENDATARIO. El renta será pago por mensualidades, realizándose el abono el día 11 de cada mes. El depósito deberá efectuar en la siguiente bancaria a nombre de ZAMORA QUISPE. El contrato será de 12 meses que se computará a partir of the fecha de suscripción. El bien como le será entregado, lo cual al final quedará a favor of the dueña, sin reembolso alguno. Para efectos que segenere con el contrato, las partes se someten a la competencia territorial de los jueces y tribunales del cono norte de Lima. El pago per mes de Setiembre del 2024 por la suma de S/ 1400 MIL CUATROCIENTOS SOLES a LA ARRendADORA. El barrio de Los Laureles N.o 450, Mz. B Lt. 20, Urb. Villa Central. El local comercial se encuentra desocupado en buen estado de conservación y habitabilidad, con piso de cerámica tipo porcelanato. El parte de Zamora QuISPE está en la obligación moral y en especial legal de gestionar su autorización de funcamiento. El Parte de VELÁSQUEZ MENDOZA, JULIO, se obliga a destinar el bien única y exclusivamente para FARMACIA. No pudiendo emplear ninguna oficina, ni como casa habitación.

**2.1. Otros modelos - Q&A**

> Con el afán de entender mejor estos textos, podemos introducir más modelos que nos permitan obtener insights de los textos. Por ejemplo, utilizaremos el modelo **mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es** para hacer preguntas y respuestas de cosas específicas en el texto



In [ ]:
qa_es = pipeline("question-answering", model="mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es")

question = "¿Cuál es la duración del contrato?"
answer = qa_es(question=question, context=full_text)

print(f"Q: {question}\nA: {answer['answer']} (score: {answer['score']:.2f})")

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Some weights of the model checkpoint at mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0


Q: ¿Cuál es la duración del contrato?
A: 12 meses (score: 0.64)


In [ ]:
qa_es = pipeline("question-answering", model="mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es")

question = "¿Cuál es la dirección del inmueble?"
answer = qa_es(question=question, context=full_text)

print(f"Q: {question}\nA: {answer['answer']} (score: {answer['score']:.2f})")

Some weights of the model checkpoint at mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


Q: ¿Cuál es la dirección del inmueble?
A: Av. Alameda del Norte N.º 451 (score: 0.11)


In [ ]:
qa_es = pipeline("question-answering", model="mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es")

question = "¿Cuál es el codigo interbancario de la arrendadora"
answer = qa_es(question=question, context=full_text)

print(f"Q: {question}\nA: {answer['answer']} (score: {answer['score']:.2f})")

Some weights of the model checkpoint at mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


Q: ¿Cuál es el codigo interbancario de la arrendadora
A: 018-00100234567890123 (score: 0.12)


**2.2. Otros modelos - Clasificador de cláusulas**

 Se agregó un modelo de clasificación de cláusulas legales de manera que una persona en vez de leer todo el texto podría irse de frente a la cláusula que le llama la atención según la clasificación que obtenga.    Este caso lo veo más aplicable para derecho. El modelo usado es **joeddav/xlm-roberta-large-xnli**

In [ ]:
classifier = pipeline("zero-shot-classification", model="joeddav/xlm-roberta-large-xnli")

# Divide usando líneas con todo mayúsculas que son típicas de encabezados legales
clausulas = re.split(r'\n(?=\s*[A-ZÑÁÉÍÓÚÜ]{3,}[^\n]*:)', full_text)

# Filtrar si hay cláusulas muy cortas o vacías
clausulas = [c.strip() for c in clausulas if len(c.strip()) > 50]

# ✅ Paso 4: Clasificar cada cláusula con Zero-Shot
resultados = []



labels = ["Pago", "Confidencialidad", "Terminación", "Obligaciones", "Jurisdicción"]

for idx, clausula in enumerate(clausulas):
    result = classifier(clausula[:1000], candidate_labels=labels)
    resultados.append({
        "cláusula": clausula.split("\n")[0].strip(),  # Título o primera línea
        "más probable": result["labels"][0],
        "score": result["scores"][0],
        "Scores": list(zip(result["labels"], result["scores"]))
    })

# ✅ Paso 5: Mostrar en tabla (puedes exportar si quieres)
import pandas as pd

df = pd.DataFrame([{
    "Cláusula": r["cláusula"],
    "Categoría detectada": r["más probable"],
    "Puntaje": round(r["score"], 2),
    "Etiquetas": ", ".join([f"{l} ({s:.2f})" for l, s in r["Scores"]])
} for r in resultados])

pd.set_option('display.max_colwidth', None)
display(df)

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


,Cláusula,Categoría detectada,Puntaje,Etiquetas
0,ARRENDAMIENTO DE LOCAL COMERCIAL,Obligaciones,0.32,"Obligaciones (0.32), Pago (0.20), Jurisdicción (0.19), Terminación (0.18), Confidencialidad (0.11)"
1,PRIMERA: LA ARRENDADORA declara ser propietaria del inmueble ubicado en la,Obligaciones,0.56,"Obligaciones (0.56), Jurisdicción (0.18), Pago (0.17), Terminación (0.09), Confidencialidad (0.01)"
2,SEGUNDA: LA ARRENDADORA deja constancia que el local comercial ubicado en,Obligaciones,0.45,"Obligaciones (0.45), Jurisdicción (0.24), Pago (0.22), Terminación (0.07), Confidencialidad (0.03)"
3,"TERCERA: LA ARRENDADORA, le hace entrega del local comercial en buen estado;",Obligaciones,0.81,"Obligaciones (0.81), Pago (0.08), Jurisdicción (0.07), Terminación (0.02), Confidencialidad (0.02)"
4,"CUARTA: Por el presente contrato, LA ARRENDADORA se obliga a ceder el uso del",Obligaciones,0.73,"Obligaciones (0.73), Pago (0.24), Terminación (0.01), Jurisdicción (0.01), Confidencialidad (0.00)"
5,QUINTA: Las partes acuerdan que el monto de la renta que pagará EL,Pago,0.82,"Pago (0.82), Obligaciones (0.10), Jurisdicción (0.06), Terminación (0.01), Confidencialidad (0.00)"
6,"SEXTA: La forma de pago de la renta será por mensualidades, realizándose el abono el",Pago,0.65,"Pago (0.65), Obligaciones (0.18), Jurisdicción (0.09), Terminación (0.07), Confidencialidad (0.02)"
7,SÉPTIMA: Las partes convienen fijar un plazo de duración determinada para el presente,Obligaciones,0.74,"Obligaciones (0.74), Terminación (0.15), Pago (0.06), Jurisdicción (0.04), Confidencialidad (0.01)"
8,OCTAVA: LA ARRENDADORA se obliga a entregar el bien objeto de la prestación a,Obligaciones,0.87,"Obligaciones (0.87), Pago (0.11), Terminación (0.01), Jurisdicción (0.01), Confidencialidad (0.00)"
9,NOVENO: EL ARRENDATARIO se obliga a pagar puntualmente el monto de la,Obligaciones,0.79,"Obligaciones (0.79), Pago (0.16), Terminación (0.03), Jurisdicción (0.02), Confidencialidad (0.01)"


**2.3. Otros modelos - Clasificador de cláusulas**

 Se agregó un modelo de clasificación de cláusulas legales de manera que una persona en vez de leer todo el texto podría irse de frente a la cláusula que le llama la atención según la clasificación que obtenga.    Este caso lo veo más aplicable para derecho. El modelo usado es **joeddav/xlm-roberta-large-xnli**

In [ ]:


# Inicializar el pipeline
ner_es = pipeline("ner", model="mrm8488/bert-spanish-cased-finetuned-ner", grouped_entities=True)

# Cortar texto seguro
chunks = textwrap.wrap(full_text, width=400)
all_entities = []

for chunk in chunks:
    try:
        ents = ner_es(chunk)
        all_entities.extend(ents)
    except Exception as e:
        print(f"❌ Error en chunk: {e}")

# Limpiar entidades
def clean_entity(e):
    word = re.sub(r"##", "", e["word"]).strip()
    word = word.strip(",.():;–-")
    return {
        "Entidad detectada": word,
        "Tipo": e["entity_group"],
        "Confianza": round(e["score"], 2)
    }

# Aplicar limpieza
ent_clean = [clean_entity(e) for e in all_entities if len(e["word"]) > 2 and not e["word"].startswith("##")]

# Quitar duplicados por entidad + tipo
df_entities = pd.DataFrame(ent_clean).drop_duplicates(subset=["Entidad detectada", "Tipo"])



'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 08bf8b5d-b9f9-4436-90a3-b70f67459b09)')' thrown while requesting HEAD https://huggingface.co/mrm8488/bert-spanish-cased-finetuned-ner/resolve/main/config.json
Retrying in 1s [Retry 1/5].


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Some weights of the model checkpoint at mrm8488/bert-spanish-cased-finetuned-ner were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/token_classification.py:186: UserWarning: `grouped_entities` is deprecated and will be removed in version v5.0.0, defaulted to `aggregation_strategy="AggregationStrategy.SIMPLE"` instead.
  warnings.warn(


NameError: name 'textwrap' is not defined

In [ ]:
# Ordenar por confianza descendente
df_entities = df_entities.sort_values(by="Confianza", ascending=False).reset_index(drop=True)

# Mostrar tabla final ordenada
pd.set_option('display.max_colwidth', None)
display(df_entities)

,Entidad detectada,Tipo,Confianza
0,Jardines del Sol,LOC,1.00
1,Conjunto Residencial Santa Clara,LOC,1.00
2,VELÁSQUEZ MENDOZA,PER,1.00
3,Distrito de Independencia,LOC,1.00
4,Los Laureles,LOC,1.00
...,...,...,...
56,PLAZO,MISC,0.56
57,1698º,MISC,0.54
58,VIGÉSIMO,MISC,0.52
59,Comercial,MISC,0.51
